#Cell 1 — Setup & Randomized Search

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score
from scipy.stats import loguniform

# Load dataset
data = load_breast_cancer()
X = data.data

# Malignant = 1, Benign = 0
y = 1 - data.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

# Parameter distributions
param_distributions = {
    "model__C": loguniform(0.001, 100),
    "model__class_weight": [None, "balanced"],
    "model__solver": ["lbfgs"]
}

# Malignant Recall scorer (Malignant = 1)
malignant_scorer = make_scorer(recall_score, pos_label=1)

# Cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=malignant_scorer,
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit
random_search.fit(X_train, y_train)

# Results
print("\nBest Parameters:", random_search.best_params_)
print("Best CV Malignant Recall:", random_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters: {'model__C': np.float64(1.2357483710912178), 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}
Best CV Malignant Recall: 0.9588235294117646


#Cell 2: Test Evaluation

In [2]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Best model
best_random_model = random_search.best_estimator_

# Prediction
y_pred_random = best_random_model.predict(X_test)

# Evaluation
print("RandomizedSearchCV — Test Results")
print("=" * 45)

print("Accuracy:", accuracy_score(y_test, y_pred_random))
print("Malignant Precision:", precision_score(y_test, y_pred_random, pos_label=1))
print("Malignant Recall:", recall_score(y_test, y_pred_random, pos_label=1))
print("Malignant F1:", f1_score(y_test, y_pred_random, pos_label=1))

print("\nConfusion Matrix (Benign=0, Malignant=1):")
print(confusion_matrix(y_test, y_pred_random, labels=[0, 1]))

RandomizedSearchCV — Test Results
Accuracy: 0.9736842105263158
Malignant Precision: 0.975609756097561
Malignant Recall: 0.9523809523809523
Malignant F1: 0.963855421686747

Confusion Matrix (Benign=0, Malignant=1):
[[71  1]
 [ 2 40]]
